# Taller: Auditoría ciudadana de la contratación pública con PySpark

## Misión: presentar evidencia ante el comité

El comité ciudadano de vigilancia de la contratación pública quiere estudiar los contratos más recientes publicados en SECOP II. Algunas personas sospechan que los recursos públicos están concentrados en pocas entidades, proveedores o modalidades de contratación.

El comité no acepta una respuesta basada únicamente en una tabla final. Quiere saber:

1. ¿Qué muestran los datos?
2. ¿Cómo se procesaron tantos registros?
3. ¿Qué ocurre detrás de PySpark cuando se filtra, se agrupa o se calcula una estadística?

Actuarás como científico o científica de datos. Tu informe debe combinar código, interpretación y una explicación de la arquitectura distribuida.

> **Importante:** este análisis busca estudiar patrones de contratación. No permite, por sí solo, afirmar que exista corrupción. Tus conclusiones deben distinguir entre evidencia, limitaciones e hipótesis.

## Conexión con el taller anterior

En el taller de MapReduce aprendiste a separar el trabajo entre `mapper`, `combiner`, `shuffle` y `reducer`. Ahora repetirás ideas parecidas usando DataFrames de PySpark:

| MapReduce | PySpark | Pregunta guía |
| --- | --- | --- |
| Mapper | `filter()`, `withColumn()` | ¿Qué filas transformamos o descartamos? |
| Shuffle por clave | `groupBy()` | ¿Cómo se reúnen los registros relacionados? |
| Combiner | agregaciones parciales internas | ¿Cómo reducimos datos antes de moverlos? |
| Reducer | `sum()`, `count()`, `avg()` | ¿Cómo consolidamos el resultado? |

La mediana será especial: no se obtiene igual que una suma o un máximo. La aproximaremos con `approxQuantile()` y explicaremos por qué.

## Entrega individual

Debes entregar:

- este notebook ejecutado;
- dos diagramas hechos a mano, fotografiados o escaneados y subidos a Blackboard;
- respuestas breves a las preguntas del comité;
- conclusiones que diferencien resultados observados de interpretaciones.

Los diagramas se entregan por separado del notebook:

- **Diagrama 1:** arquitectura distribuida de la media;
- **Diagrama 2:** arquitectura conceptual de la mediana aproximada.

## Volumen de trabajo

Trabajaremos con hasta **1.000.000 de contratos**. La consulta seleccionará únicamente las columnas necesarias y devolverá los contratos más recientes en un orden reproducible.

Si una celda tarda demasiado o el entorno se queda sin memoria, cambiaremos `TAMANO_MUESTRA` por `100_000` o `10_000`. En ese caso, debes registrar en una respuesta qué tamaño utilizaste y cómo puede afectar la interpretación.

La muestra es grande para estudiar procesamiento de datos, pero no representa automáticamente los 6 millones de filas de SECOP II ni constituye una muestra aleatoria.

In [1]:
import pandas as pd
import requests

URL = "https://www.datos.gov.co/api/v3/views/jbjy-vk9h/query.json"
TAMANO_MUESTRA = 100_000

# Si el entorno se vuelve lento, prueba con 100_000 o 10_000.
columnas = [
    "nombre_entidad",
    "departamento",
    "ciudad",
    "sector",
    "id_contrato",
    "estado_contrato",
    "descripcion_del_proceso",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "fecha_de_firma",
    "valor_del_contrato",
    "proveedor_adjudicado",
    "codigo_entidad",
    "codigo_proveedor",
]

consulta = {
    "query": (
        "SELECT " + ", ".join(columnas)
        + " ORDER BY fecha_de_firma DESC, id_contrato ASC"
        + f" LIMIT {TAMANO_MUESTRA}"
    ),
    "page": {
        "pageNumber": 1,
        "pageSize": TAMANO_MUESTRA,
    },
    "includeSynthetic": False,
}

respuesta = requests.post(URL, json=consulta)
respuesta.raise_for_status()
tabla = pd.DataFrame(respuesta.json())

print(f"Filas descargadas: {len(tabla):,}")
print(f"Columnas descargadas: {len(tabla.columns)}")
tabla.head()

Filas descargadas: 100,000
Columnas descargadas: 13


,nombre_entidad,departamento,ciudad,sector,id_contrato,estado_contrato,descripcion_del_proceso,tipo_de_contrato,modalidad_de_contratacion,valor_del_contrato,proveedor_adjudicado,codigo_entidad,codigo_proveedor
0,ESTABLECIMIENTO CARCELARIO PURIFICACION,No Definido,No Definido,Ley de Justicia,CO1.PCCNTR.1000002,Borrador,Sin Descripcion,Suministros,Mínima cuantía,1200000,OLGA YANET ROJAS BARBOSA,702957242,704055086
1,EMPRESA DE ACUEDUCTO Y ALCANTARILLADO DE BOGOT...,No Definido,No Definido,Industria,CO1.PCCNTR.10001,Borrador,SUMINISTRO DE QUIMICOS PARA EL MANTENIMIENTO I...,Suministros,Contratación régimen especial,0,QSM CHEMICALS AND SERVICES SAS,700064108,700474067
2,ESM BATALLÓN ASPC NO. 6,Tolima,Ibagué,Servicio Público,CO1.PCCNTR.1000201,enviado Proveedor,CONTRATAR LA PRESTACION DEL SERVICIO DE ASEO Y...,Prestación de servicios,Selección Abreviada de Menor Cuantía,133000000,KIOS S.A.S. BIC,702962226,701231128
3,ESTABLECIMIENTO PENITENCIARIO DE MEDIANA SEGUR...,Nariño,Tumaco,Ley de Justicia,CO1.PCCNTR.1000202,En aprobación,CONTRATAR EL SUMINISTRO DE BEBIDAS GASEOSAS E ...,Suministros,Mínima cuantía,2404123.54,MONTALVO DIAZ DALLYS ANDREA,703706218,704085075
4,ESTABLECIMIENTO PENITENCIARIO DE MEDIANA SEGUR...,Nariño,Tumaco,Ley de Justicia,CO1.PCCNTR.1000301,En aprobación,CONTRATAR EL SUMINISTRO DE PRODUCTOS ALIMENTIC...,Suministros,Mínima cuantía,4369500,EDISNAR,703706218,704023076


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Auditoria SECOP II") \
    .getOrCreate()

spark_df = spark.createDataFrame(tabla)

print(f"Filas en PySpark: {spark_df.count():,}")
spark_df.show(5, truncate=False)
spark_df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/12 16:29:27 WARN Utils: Your hostname, codespaces-ee3e1f, resolves to a loopback address: 127.0.0.1; using 10.0.11.188 instead (on interface eth0)
26/09/12 16:29:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/12 16:29:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conver

+--------------------+-----------+--------------------+--------------+--------------------+-----------+--------------------+--------------------+--------------------+-----------------+------------------+-----------------------+---------------+-----------------------------+-----------------------+--------------------+-------------------------+--------------------------+--------------------+----------------------------+-------------------------+----------------------+--------------------+-------------------+--------------------+--------+-------+------------------------+-----------+--------------------+------------------------+---------+----------------------+--------------+------------------+------------------------+---------------+-----------------------+------------+----------------+------------------+----------------------------+----------+--------------+---------------+----------------+------------------+-------------------+--------------------+--------------------------+------------

## Cómo pensar en un DataFrame de PySpark

Un DataFrame representa una tabla distribuida. La tabla puede estar repartida entre varias particiones y las operaciones se describen antes de ejecutarse.

En este taller usaremos principalmente la sintaxis funcional con referencias a columnas:

```python
spark_df.filter(spark_df.estado_contrato == "En ejecución")
```

También existe una sintaxis equivalente con `col()`:

```python
spark_df.filter(col("estado_contrato") == "En ejecución")
```

La primera forma ayuda a visualizar que una columna pertenece al DataFrame. La segunda es útil cuando el nombre se construye dinámicamente o cuando una columna contiene caracteres especiales. Aquí practicaremos ambas, pero las soluciones principales usarán el estilo funcional.

Antes de ejecutar cada bloque, escribe una predicción: ¿cuántas filas esperas?, ¿qué columnas cambiarán?, ¿qué operación equivale al mapper o al reducer?

## Ejercicio 1. Reconocer la evidencia

El comité necesita saber qué información contiene cada contrato antes de formular una conclusión.

### Tu misión

1. Compara `tabla.head()` con `spark_df.show()`.
2. Inspecciona el esquema y localiza las columnas de identificación, tiempo y valor.
3. Selecciona con PySpark únicamente `nombre_entidad`, `modalidad_de_contratacion`, `fecha_de_firma` y `valor_del_contrato`.
4. Repite la selección usando una referencia con punto para al menos una columna.
5. Comprueba que seleccionar columnas no cambia el número de filas.

### Pregunta del comité

¿Por qué seleccionar menos columnas puede ser importante cuando los datos están distribuidos?

### Repaso de MapReduce

Todavía no hay agrupación. Piensa en esta operación como una preparación del registro que recibe cada mapper.

In [ ]:
# Escribe aquí tu solución del Ejercicio 1.
# Usa select(), show() y count().
# No copies una solución de otra persona: primero escribe tu predicción.

# columnas_ejercicio_1 = spark_df.select(...)
# columnas_ejercicio_1.show(5, truncate=False)
# print("Filas originales:", spark_df.count())
# print("Filas seleccionadas:", columnas_ejercicio_1.count())

## Ejercicio 2. Limpieza y filtro forense

El comité advierte que una suma de contratos solo es defendible si el valor puede interpretarse como un número. En los datos provenientes de APIs, los tipos no siempre llegan como esperamos.

### Tu misión

1. Convierte `valor_del_contrato` a un tipo numérico.
2. Crea `valor_millones` dividiendo el valor entre un millón.
3. Elimina o filtra registros sin valor utilizable.
4. Conserva únicamente contratos con valor positivo.
5. Compara el número de filas antes y después.

Usa `withColumn()`, `cast()`, `isNotNull()` y `filter()`.

### Pregunta del comité

¿Qué riesgo tendría calcular una suma antes de revisar el tipo y los valores nulos?

### Repaso de MapReduce

Este trabajo se parece al mapper: cada registro se transforma o se descarta de forma independiente antes de la consolidación. Explica por qué filtrar temprano reduce el volumen que debe viajar hacia una fase de agrupación.

In [ ]:
# Escribe aquí tu solución del Ejercicio 2.
# Puedes crear una nueva variable en cada transformación.
# Recuerda que los DataFrames de PySpark son inmutables.

# contratos_limpios = spark_df.withColumn(...)
# contratos_limpios = contratos_limpios.filter(...)
# contratos_limpios.show(5, truncate=False)
# print("Filas después de limpiar:", contratos_limpios.count())

## Ejercicio 3. ¿Dónde se concentran los recursos?

El comité quiere una primera respuesta cuantitativa:

> ¿Qué entidades concentran el mayor valor contratado y cuántos contratos representan?

### Tu misión

Usando `contratos_limpios`:

1. Agrupa por `nombre_entidad`.
2. Calcula la cantidad de contratos.
3. Calcula el valor total contratado.
4. Ordena el resultado de mayor a menor valor total.
5. Muestra las diez entidades principales.

Usa `groupBy()`, `agg()`, `count()`, `sum()`, `alias()` y `orderBy()`.

Después repite el análisis agrupando por `modalidad_de_contratacion`.

### Preguntas del comité

- ¿Una entidad con mayor valor total necesariamente tiene contratos más grandes?
- ¿Qué diferencia hay entre concentración por cantidad de contratos y concentración por valor?
- ¿Qué limitación introduce trabajar únicamente con los contratos más recientes?

### Repaso de MapReduce

Escribe debajo de tu código qué representa cada elemento:

- clave del mapper;
- agrupación del shuffle;
- operación del reducer;
- posible trabajo del combiner.

No basta con decir que `groupBy()` es “como un reducer”: explica qué información se mueve y qué información se consolida.

In [ ]:
# Escribe aquí tu solución del Ejercicio 3.
# Usa la variable contratos_limpios creada en el ejercicio anterior.

# resumen_entidades = (
#     contratos_limpios
#     .groupBy(...)
#     .agg(
#         count("*").alias(...),
#         sum(...).alias(...),
#     )
#     .orderBy(desc(...))
# )
# resumen_entidades.show(10, truncate=False)

## Ejercicio 4. La media no es una caja negra

El comité pregunta por el valor promedio de los contratos agrupados por modalidad. También exige una explicación de cómo se puede calcular esa media cuando los datos están repartidos.

### Tu misión con PySpark

1. Calcula la media del valor del contrato por modalidad.
2. Incluye también la cantidad de contratos y el valor total.
3. Ordena las modalidades por valor promedio.
4. Interpreta una modalidad con pocos contratos y un valor promedio muy alto.

Usa `avg()` y compárala con la estrategia de `(suma, conteo)` que implementaste en MapReduce.

### Entrega manuscrita 1: arquitectura de la media

En una hoja de papel dibuja y fotografía una arquitectura con varias particiones. Debe mostrar:

- registros separados en al menos tres particiones;
- suma parcial y conteo parcial en cada partición;
- combinación de esos resultados;
- fórmula de la media global;
- ubicación conceptual del mapper, combiner, shuffle y reducer.

Incluye una frase que explique por qué **no** se deben promediar directamente las medias locales cuando las particiones tienen tamaños diferentes.

Sube esta evidencia por separado a Blackboard y menciona en el notebook el nombre del archivo entregado.

In [ ]:
# Escribe aquí tu solución del Ejercicio 4.
# Compara la media por modalidad con el resultado de MapReduce.

# resumen_modalidades = (
#     contratos_limpios
#     .groupBy(...)
#     .agg(
#         count("*").alias(...),
#         sum(...).alias(...),
#         avg(...).alias(...),
#     )
#     .orderBy(desc(...))
# )
# resumen_modalidades.show(truncate=False)

# Escribe aquí el nombre del archivo que subirás a Blackboard:
# evidencia_media = "..."

## Ejercicio 5. La batalla de la mediana

La media puede quedar dominada por unos pocos contratos extraordinariamente grandes. El comité solicita una medida de posición más resistente:

> ¿Cuál es la mediana aproximada del valor de los contratos y cómo se compara con la media?

### Tu misión con PySpark

1. Calcula la mediana aproximada del valor del contrato con `approxQuantile()`.
2. Usa el percentil `0.5`.
3. Define una tolerancia de error y explica qué significa.
4. Compara la mediana con la media calculada en el ejercicio anterior.
5. Decide cuál de las dos medidas describe mejor un contrato “típico” en este contexto.

### Entrega manuscrita 2: arquitectura de la mediana aproximada

En una hoja diferente dibuja una arquitectura conceptual inspirada en la actividad de los post-its. Debe mostrar:

- datos divididos en particiones;
- un resumen local por partición, como rangos, frecuencias o cuantiles aproximados;
- combinación de resúmenes;
- estimación del percentil 0.5;
- diferencia frente a ordenar todos los contratos y tomar el valor central.

Responde en el mismo archivo:

- ¿Por qué no basta con promediar las medianas locales?
- ¿Por qué la mediana no se comporta como una suma o un máximo?
- ¿Por qué el resultado puede ser aproximado?
- ¿Qué relación tiene esta arquitectura con la actividad de MapReduce realizada con post-its?

Sube esta segunda evidencia por separado a Blackboard.

In [ ]:
# Escribe aquí tu solución del Ejercicio 5.
# approxQuantile recibe el nombre de la columna, los percentiles y una tolerancia.
# Primero decide qué columna y qué tolerancia son apropiadas.

# mediana_aproximada = contratos_limpios.approxQuantile(
#     ...,
#     [...],
#     ...,
# )
# print("Mediana aproximada:", mediana_aproximada)

# Escribe aquí el nombre del segundo archivo que subirás a Blackboard:
# evidencia_mediana = "..."

## Defensa final ante el comité

Escribe una respuesta breve, con tus propias palabras, para cada pregunta:

1. ¿Qué parte del análisis se parece al mapper?
2. ¿Dónde aparece el agrupamiento equivalente al shuffle?
3. ¿Qué operación funciona como reducer en el análisis de concentración?
4. ¿Por qué una suma puede combinarse exactamente y una mediana suele aproximarse?
5. ¿Qué hallazgo observaste sobre la concentración de contratación?
6. ¿Qué limitaciones tienen los datos, el orden de la muestra y el tamaño elegido?
7. Si el comité afirmara “la entidad con mayor valor total es necesariamente la más sospechosa”, ¿qué responderías?

### Criterio de calidad

Una buena respuesta no solo muestra una cifra. Identifica el universo analizado, explica cómo se obtuvo, reconoce incertidumbres y evita convertir una correlación o concentración en una acusación.

## Referencia de sintaxis

| Necesidad | pandas | PySpark usado en este taller |
| --- | --- | --- |
| Ver registros | `tabla.head()` | `spark_df.show()` |
| Revisar tipos | `tabla.dtypes` | `spark_df.printSchema()` |
| Seleccionar columnas | `tabla[[...]]` | `select()` |
| Filtrar filas | `tabla[condicion]` | `filter()` |
| Crear columna | asignación directa | `withColumn()` |
| Agrupar | `groupby()` | `groupBy()` |
| Agregar | `sum()`, `mean()` | `sum()`, `avg()`, `count()` |
| Mediana | `median()` | `approxQuantile()` |

Conserva esta tabla como mapa de transición, pero basa tus conclusiones en el código que construiste y en las dos arquitecturas que entregaste.